# EfficientDet-Lite0 Training for SMD Component Detection

Train a model to detect electronic components on tape/reel using the smdComponents dataset.
Output: `efficientdet-lite0.tflite` — drop-in replacement for the Component Counter Android app.

---

In [ ]:
# @title Install dependencies
!pip install -q tflite-model-maker tensorflow==2.16.0
!pip install -q pycocotools
!pip install -q roboflow

import os
import numpy as np
import tensorflow as tf
print(f"TensorFlow version: {tf.__version__}")

In [ ]:
# @title Download dataset from Roboflow
from roboflow import Roboflow

# === CONFIG ===
ROBOFLOW_API_KEY = ""  # @param {type:"string"}
ROBOFLOW_WORKSPACE = "dainius"  # @param
ROBOFLOW_PROJECT = "smdcomponents"  # @param
ROBOFLOW_VERSION = 2  # @param {type:"integer"}  (v2 = raw images, v6 = augmented)
MODEL_NAME = "efficientdet-lite0"  # @param ["efficientdet-lite0", "efficientdet-lite1", "efficientdet-lite2"]
BATCH_SIZE = 8  # @param {type:"integer"}
EPOCHS = 50  # @param {type:"integer"}

if not ROBOFLOW_API_KEY:
    raise ValueError("Enter your Roboflow API key! Get it from https://app.roboflow.com/settings/api")

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
dataset = project.version(ROBOFLOW_VERSION).download("tfrecord")

print(f"Dataset downloaded to: {dataset.location}")

In [ ]:
# @title Download dataset directly (fallback if roboflow pip fails)
# Alternative: Download TFRecord directly from Roboflow export URL
# The URL structure is:
# https://universe.roboflow.com/ds/{EXPORT_ID}?key={API_KEY}
# Get your export ID from the dataset page → Export → TFRecord

TF_MANUAL_URL = ""  # @param {type:"string"}

if TF_MANUAL_URL and not os.path.exists("smdcomponents_dataset"):
    !wget -O dataset.zip "{TF_MANUAL_URL}"
    !unzip -q dataset.zip -d smdcomponents_dataset
    print("Dataset extracted to smdcomponents_dataset/")
    dataset.location = "smdcomponents_dataset"

## Train EfficientDet-Lite0

Using TensorFlow Model Maker's object detection pipeline.

In [ ]:
# @title Train model
from tflite_model_maker import configs
from tflite_model_maker import ExportFormat
from tflite_model_maker import model_spec
from tflite_model_maker import object_detector

# Load model spec
spec = model_spec.get(MODEL_NAME)
print(f"Using model spec: {MODEL_NAME}")

# Load dataset
train_data, validation_data, test_data = object_detector.DataLoader.from_tfrecord(
    train_dir=os.path.join(dataset.location, "train"),
    validation_dir=os.path.join(dataset.location, "valid"),
    test_dir=os.path.join(dataset.location, "test"),
    label_map={
        1: "Resistor",
        2: "Diode",
        3: "Transistor",
        4: "Condensator"
    }
)
print(f"Train: {len(train_data)}, Validation: {len(validation_data)}, Test: {len(test_data)}")

# Configure training
config = configs.ObjectDetectionConfig(
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    learning_rate=0.005,
    lr_decay_steps=20,
    lr_decay_rate=0.96,
    cosine_lr=True
)

# Train
model = object_detector.create(
    train_data=train_data,
    model_spec=spec,
    validation_data=validation_data,
    config=config
)

# Evaluate
eval_result = model.evaluate(test_data)
print(f"Test mAP: {eval_result['mAP']}")

In [ ]:
# @title Export TFLite model
export_dir = "/content/component-counter-model"
os.makedirs(export_dir, exist_ok=True)

# Export to TFLite
model.export(export_dir=export_dir, export_format=ExportFormat.TFLITE)

# Find the exported .tflite file
import glob
tflite_files = glob.glob(os.path.join(export_dir, "*.tflite"))
if tflite_files:
    model_path = tflite_files[0]
    print(f"Model exported: {model_path}")
    
    # Copy to root with standard name
    !cp "{model_path}" /content/efficientdet-lite0.tflite
    print("Copied to: /content/efficientdet-lite0.tflite")
    
    # Show model size
    size_mb = os.path.getsize("/content/efficientdet-lite0.tflite") / (1024 * 1024)
    print(f"Model size: {size_mb:.2f} MB")
    
    # Download
    from google.colab import files
    files.download("/content/efficientdet-lite0.tflite")
else:
    print("No .tflite found. Check export format.")

In [ ]:
# @title Convert to full-integer quantized (optional, smaller/faster)
# EfficientDet-Lite models already use post-training quantization.
# This step converts to INT8 quantized for maximum speed.

converter = tf.lite.TFLiteConverter.from_saved_model(
    os.path.join(export_dir, "saved_model")
)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]

tflite_fp16 = converter.convert()
with open("/content/efficientdet-lite0-fp16.tflite", "wb") as f:
    f.write(tflite_fp16)

size_fp32 = os.path.getsize("/content/efficientdet-lite0.tflite") / (1024 * 1024)
size_fp16 = os.path.getsize("/content/efficientdet-lite0-fp16.tflite") / (1024 * 1024)
print(f"FP32: {size_fp32:.2f} MB → FP16: {size_fp16:.2f} MB")

## Deploy to Android

1. Download `efficientdet-lite0.tflite` from above
2. Place it in `app/src/main/assets/` of the Component Counter project
3. Update `ObjectDetectorHelper.kt`:
   - Change `modelName` from `mobilenetv1.tflite` to `efficientdet-lite0.tflite`
   - Adjust `maxResults` and `threshold` if needed
4. Rebuild the app: `./gradlew assembleDebug`

The TFLite Task Vision ObjectDetector API is the same for both models.
EfficientDet-Lite0 is a drop-in replacement.

## Label Map for App

The model outputs class IDs:
```
1 → Resistor
2 → Diode
3 → Transistor
4 → Condensator (Capacitor)
```

The ObjectDetector returns `Detection` objects with `categories` containing
the class index and score.